# MEG-BAR — Colab Runner (No Drive Mount)

Runs the **MEG-BAR Phase 1 robust evaluation** on Colab T4 GPU.

### Instructions:
1. **Runtime → Change runtime type → T4 GPU** (required; CPU will OOM).
2. Upload `MEGBAR_colab_payload.zip` using the file browser (left panel) or run the upload cell below.
3. Run all cells sequentially.

In [ ]:
import os, zipfile, shutil
from google.colab import files

# 1. Upload payload zip if not already present
if not os.path.exists('MEGBAR_colab_payload.zip'):
    print('Please upload MEGBAR_colab_payload.zip now:')
    uploaded = files.upload()

# 2. Unzip into working directory
RUN = '/content/megbar_run'
if os.path.isdir(RUN):
    shutil.rmtree(RUN)
os.makedirs(RUN, exist_ok=True)

with zipfile.ZipFile('MEGBAR_colab_payload.zip', 'r') as zf:
    zf.extractall(RUN)

# Verify structure
print('Extracted to', RUN)
print('Scripts:', os.listdir(os.path.join(RUN, 'scripts', 'real')))
print('Data files:', len(os.listdir(os.path.join(RUN, 'data', 'real'))), 'files')
print('Results:', os.listdir(os.path.join(RUN, 'results')))

In [ ]:
import torch, sys, subprocess, os

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: No GPU detected. Enable T4 in Runtime settings.')

os.chdir('/content/megbar_run')
env = dict(os.environ, PYTHONIOENCODING='utf-8')

print('\nStarting robust evaluation pipeline...\n', flush=True)
# Run from scripts/real where 08_robust_eval.py lives and expects data/real relative path
rc = subprocess.run([sys.executable, 'scripts/real/08_robust_eval.py'], 
                    cwd='/content/megbar_run', env=env).returncode
print('\nFinished with exit code:', rc)

In [ ]:
# Download results back to your machine
from google.colab import files
res_path = '/content/megbar_run/data/real/robust_results.json'
rep_path = '/content/megbar_run/data/real/robust_report.md'
if os.path.exists(res_path):
    files.download(res_path)
    print('Downloaded:', res_path)
if os.path.exists(rep_path):
    files.download(rep_path)
    print('Downloaded:', rep_path)
if not (os.path.exists(res_path) or os.path.exists(rep_path)):
    print('No result files found yet — check the run output above for errors.')